# Quadratic examples and plots (2D + 3D)

This notebook exercises the quadratic primitives

- **2D** - heatmap on `(x, y)` at `t = 0` (`method="2D"`)
- **3D** - isosurface on `(x, y, yaw)` at `t = 0` (`method="3D"`)

Builders are `AppliedSet` aliases. Geometry lives on `QuadraticSetImpl` (`ball` / `cylinder` / `ellipsoid` delegate to `quadratic`). `TVHJImpl` implements `quadratic`.

| Sets             | Description                                      |
|------------------|--------------------------------------------------|
| `BallSet`        | `||x-c||^2 - r^2` in the chosen axes             |
| `CylinderSet`    | distance to an axis, still a quadratic           |
| `EllipsoidSet`   | `sum ((x_i-c_i)/r_i)^2 - 1`                      |
| `QuadraticSet`   | general `(x-c)^T A (x-c) + b^T (x-c) + c`        |


In [9]:
from math import pi

import numpy as np

from pyspect import *
from pyspect.impls.hj_reachability import TVHJImpl
from pyspect.systems.hj_reachability import Air3d

AXES = [
    dict(name="t", bounds=[0, 1], step=1.0, unit="s"),
    dict(name="x", bounds=[-5, +5], points=25, unit="m"),
    dict(name="y", bounds=[-5, +5], points=25, unit="m"),
    dict(name="yaw", bounds=[-pi, +pi], points=12, unit="rad"),
]

impl = TVHJImpl(dict(cls=Air3d), AXES)
SLICE_T = [("t", 0)]
CAMERA_3D = impl.PLOT.EYE_ML_SW


def maxdiff(a, b) -> float:
    return float(np.max(np.abs(np.asarray(a) - np.asarray(b))))


def show_2d(out, title: str, *, colorscale: str = "Greens") -> None:
    impl.plot(
        out,
        method="2D",
        axes=("x", "y"),
        transform_select=SLICE_T,
        layout_title=f"[2D] {title}",
        colorscale=colorscale,
        layout_height=360,
        layout_width=480,
    ).show()


def show_3d(out, title: str, *, colorscale: str = "Greens") -> None:
    impl.plot(
        out,
        method="3D",
        axes=("x", "y", "yaw"),
        transform_select=SLICE_T,
        layout_title=f"[3D] {title}",
        colorscale=colorscale,
        camera_eye=CAMERA_3D,
        layout_height=480,
        layout_width=640,
    ).show()


## 1. BallSet - disk of radius 2 in `(x, y)`

Compared to `QuadraticSet` with `A = I` and `c = -r^2`.

**3D** uses the same radius on `(x, y, yaw)` so the 0-level is a closed surface. A disk in `(x, y)` only is constant in `yaw`; Plotly `Isosurface` then stays empty.


In [10]:
S = BallSet(center=[0.0, 0.0], radius=2.0, axes=["x", "y"])
ref = QuadraticSet(
    A=[[1.0, 0.0], [0.0, 1.0]],
    c=-4.0,
    center=[0.0, 0.0],
    axes=["x", "y"],
)

out = S(impl)
print("shape", tuple(np.asarray(out).shape), "grid", impl.shape)
print(f"max |ball - quadratic(I, -r^2)| = {maxdiff(out, ref(impl))}")
show_2d(out, "ball: r=2 at origin", colorscale="Greens")

S3 = BallSet(center=[0.0, 0.0, 0.0], radius=2.0, axes=["x", "y", "yaw"])
out3 = S3(impl)
print("3D volume", np.asarray(impl.transform_to_isosurface(out3, axes=("x", "y", "yaw"), select=SLICE_T)).shape)
show_3d(out3, "ball: r=2 in (x, y, yaw)", colorscale="Greens")


shape (2, 25, 25, 12) grid (2, 25, 25, 12)
max |ball - quadratic(I, -r^2)| = 0.0


3D volume (25, 25, 12)


## 2. CylinderSet - axis along `y`, radius 1

**2D** (`axes=["x","y"]`): strip `|x| <= 1`.

**3D** (`axes=["x","y","yaw"]`, same axis along `y`): circular cylinder `x^2 + yaw^2 <= r^2`. Plotly needs a level set that varies in two of the plotted axes; the 2D strip extruded in yaw is two planes and the isosurface stays empty.


In [11]:
S2 = CylinderSet(
    center=[0.0, 0.0],
    radius=1.0,
    vector=[0.0, 1.0],
    axes=["x", "y"],
)
ref2 = QuadraticSet(
    A=[[1.0, 0.0], [0.0, 0.0]],
    c=-1.0,
    center=[0.0, 0.0],
    axes=["x", "y"],
)
out2 = S2(impl)
print(f"max |2D cylinder - quadratic| = {maxdiff(out2, ref2(impl))}")
show_2d(out2, "cylinder: along y, r=1 (strip |x|<=1)", colorscale="Oranges")

S3 = CylinderSet(
    center=[0.0, 0.0, 0.0],
    radius=1.5,
    vector=[0.0, 1.0, 0.0],
    axes=["x", "y", "yaw"],
)
ref3 = QuadraticSet(
    A=[[1.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 1.0]],
    c=-2.25,
    center=[0.0, 0.0, 0.0],
    axes=["x", "y", "yaw"],
)
out3 = S3(impl)
print(f"max |3D cylinder - quadratic| = {maxdiff(out3, ref3(impl))}")
print("shape", tuple(np.asarray(out3).shape), "grid", impl.shape)
show_3d(out3, "cylinder: along y, r=1.5 (x^2 + yaw^2)", colorscale="Oranges")


max |2D cylinder - quadratic| = 0.0


max |3D cylinder - quadratic| = 0.0
shape (2, 25, 25, 12) grid (2, 25, 25, 12)


## 3. EllipsoidSet - rx=3, ry=1.5

**2D** in `(x, y)`. **3D** adds a yaw radius so the isosurface is a closed ellipsoid (same Plotly limit as the extruded disk).


In [12]:
S = EllipsoidSet(center=[0.0, 0.0], radii=[3.0, 1.5], axes=["x", "y"])
ref = QuadraticSet(
    A=[[1.0 / 9.0, 0.0], [0.0, 1.0 / 2.25]],
    c=-1.0,
    center=[0.0, 0.0],
    axes=["x", "y"],
)
out = S(impl)
print(f"max |ellipsoid - quadratic| = {maxdiff(out, ref(impl))}")
show_2d(out, "ellipsoid: rx=3, ry=1.5", colorscale="Purples")

S3 = EllipsoidSet(center=[0.0, 0.0, 0.0], radii=[3.0, 1.5, 2.0], axes=["x", "y", "yaw"])
out3 = S3(impl)
print("3D volume", np.asarray(impl.transform_to_isosurface(out3, axes=("x", "y", "yaw"), select=SLICE_T)).shape)
show_3d(out3, "ellipsoid: rx=3, ry=1.5, ryaw=2", colorscale="Purples")


max |ellipsoid - quadratic| = 0.0


3D volume (25, 25, 12)
